[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LizbethMG-Teaching/pose2behav-book/blob/main/notebooks/analysis_multi-animal.ipynb)

# 📓 Notebook 3 – Analysis of multi animal (top-view mouse)
## 1. Introduction & objectives

In this notebook, you will analyze pose estimation outputs generated with the SuperAnimal ModelZoo on a top view multi animal video containing several mice.

**Learning goals:**

After this notebook, you should be able to:
- Load and preprocess multi animal pose data from SuperAnimal DLC
- Implement your own filtering and interpolation choices
- Compute activity and social metrics per mouse
- Integrate the results into a summary table

--- 

**About this notebook**

In this notebook, you will analyze pose-estimation data from freely-moving mice. 

# 🐭🐭🏠🎥 The Mouse House: multi animal pose challenge

<img src="https://raw.githubusercontent.com/LizbethMG-Teaching/pose2behav-book/main/assets/illustrations/cover-mice.png" width="50%">

Three mice stay together in the "Mouse House" 🐭🤍🏠, a fully monitored arena.
Every movement is tracked with SuperAnimal DeepLabCut.

Your task is to use pose data to build a behavioral profile for each mouse:
- Who is the Hyperactive One?
- Who is the Social Butterfly?
- Who is the Lone Wolf?
- And who wins each "medal" category?

🥇🥈🥉 At the end, you will assign gold, silver, and bronze medals in:
- Activity
- Sociability

For this exercise, you will work mostly independently, but everyone must create the same output variable names and structure so we can compare results.

--- 
**Instructions**

This notebook mixes pre-filled code cells (ready to run) and coding exercises that you will complete.

- Some cells are already complete (just run them).
- You are free to choose methods, but you must respect:
  - Input: the provided pose file
  - Output variable names and column names as indicated. 

👉 Here’s how to work through it:
1. Read carefully each section before running the cells.
2. When a cell requires you to code, you’ll see a TODO comment.

⚡ After finishing the course, feel free to experiment and modify the notebook as you like!

---

<img src="https://raw.githubusercontent.com/LizbethMG-Teaching/pose2behav-book/main/assets/single-frame-3-animals.png" width="50%">

## Arena geometry and scale

**‼️ Useful reference information for this LAB**

- Frame resolution: 652 × 636 pixels
- Frame rate: 66 frames per second
- Mouse body length (nose to base of tail): 80 pixels
- Real body length: 9 cm
- Approximate scale: 1 cm ≈ 8.89 pixels
- Pixel to centimeter conversion: 1 pixel ≈ 0.1125 cm

Arena geometry

- Arena shape: circular
- Diameter in the image: 460 pixels
- Real-world diameter: about 52 cm
- For simplified calculations, the arena can be approximated as a 460 × 460 pixel square
- Upper left corner of this square: x = 108, y = −78 (image coordinate system)

---

## 2. Data Loading & Format Inspection

### 2.1 Download data (prefilled)

**📋 Instructions:**
- Run the code cell below to download the dataset file.

In [ ]:
# PREFILLED, NO NEED TO CHANGE, JUST RUN THIS CELL
# Install and import the required libraries:
!pip -q install gdown tables

import os
from pathlib import Path
import gdown, pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import groupby
import re
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle


# --------------------------------------------------------------

# Detect if running in Google Colab
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    DEST = Path("/content/cleaned_pose_downloaded.h5")
else:
    DEST = Path("cleaned_pose_downloaded.h5")  # save in current folder locally
print("Saving to:", DEST)

# Select here the experiment you want to download, comment the others:
# mice-5_5min

# File : "cleaned_pose_3mice_5min.h5"
# FILE_ID = "1BxdhvhPl-2Yb39v1_ZICi9fzp_wv8u8q"

# File : "cleaned_pose_3mice_4min.h5"
FILE_ID = "1x-aevTfg0PbwwUN2y8bFJ6l5_oNK5Chf"
URL = f"https://drive.google.com/uc?id={FILE_ID}"

print("Downloading from Drive...")
_ = gdown.download(URL, str(DEST), quiet=False)

# Basic checks
assert DEST.exists() and DEST.stat().st_size > 0, "❌ Download failed or empty file."
print(f"✅ Downloaded to {DEST} ({DEST.stat().st_size/1_000_000:.2f} MB)")

# --- Load the cleaned H5 file into a pandas DataFrame ---
df = pd.read_hdf(DEST, key="df_with_missing")

print("✅ Data loaded successfully!")
print("Shape:", df.shape)
print("Columns:", list(df.columns)[:8], "...")

### 2.1 Download data (prefilled)

👉🏼 Run the next cell to explore the cleaned file you loaded.

🤓 You have a little helper that you can use to get the data for a single mouse.

In [ ]:
# 👉🏼 PREFILLED CELL — JUST RUN IT

# Show the first few columns
print("\nFirst 10 columns:")
print(df.columns[:10])

# List animals present in this file
animals = sorted(set(col.split("_")[0] for col in df.columns))
print("\nAnimals in this file:", animals)

# Show how many columns each mouse has
print("\nNumber of columns per mouse:")
for a in animals:
    count = sum(col.startswith(a + "_") for col in df.columns)
    print(f"{a}: {count} columns")

# Function to extract a single mouse
def get_mouse(df, animal_id):
    """Returns a DataFrame with only one mouse's data and clean column names."""
    cols = [c for c in df.columns if c.startswith(f"{animal_id}_")]
    dfa = df[cols].copy()
    dfa.columns = [c.replace(f"{animal_id}_", "") for c in cols]
    return dfa

# Example: extract the first mouse
example_mouse = animals[0]
mouse_df = get_mouse(df, example_mouse)

print(f"\nExample: data for {example_mouse}")
print(mouse_df.head())

In [ ]:
# Constants from the description
FRAME_RATE = 66  # frames per second
PIXEL_TO_CM = 0.1125

ARENA_DIAM_PX = 460
ARENA_CENTER_X = 108 + ARENA_DIAM_PX / 2
ARENA_CENTER_Y = -78 + ARENA_DIAM_PX / 2
ARENA_RADIUS_PX = ARENA_DIAM_PX / 2

print("Pixel to cm:", PIXEL_TO_CM)
print("Arena center (px):", ARENA_CENTER_X, ARENA_CENTER_Y)

In [ ]:
# Reset index to ensure a frame column
if "frame" in df.columns:
    df_reset = df.reset_index(drop=True)
else:
    df_reset = df.reset_index().rename(columns={"index": "frame"})

if "frame" not in df_reset.columns:
    df_reset["frame"] = np.arange(len(df_reset))

preferred_bodyparts = ["center", "bodycenter", "centroid", "spine_mid", "nose"]

records = []

for animal in animals:
    cols = [c for c in df_reset.columns if c.startswith(f"{animal}_")]
    stripped = [c.replace(f"{animal}_", "") for c in cols]

    bodyparts = set()
    for name in stripped:
        if name.endswith("_x") or name.endswith("_y") or name.endswith("_likelihood"):
            bp = name.rsplit("_", 1)[0]
            bodyparts.add(bp)
    bodyparts = sorted(list(bodyparts))
    if not bodyparts:
        print(f"No body parts found for {animal}, skipping")
        continue

    chosen_bp = None
    for bp_pref in preferred_bodyparts:
        if bp_pref in bodyparts:
            chosen_bp = bp_pref
            break
    if chosen_bp is None:
        chosen_bp = bodyparts[0]

    x_col = f"{animal}_{chosen_bp}_x"
    y_col = f"{animal}_{chosen_bp}_y"
    l_col = f"{animal}_{chosen_bp}_likelihood"

    print(f"{animal}: using body part '{chosen_bp}' with columns {x_col}, {y_col}, {l_col}")

    sub = df_reset[["frame", x_col, y_col, l_col]].copy()
    sub.columns = ["frame", "x", "y", "likelihood"]
    sub["id"] = animal

    lik_threshold = 0.9
    good = sub["likelihood"] >= lik_threshold
    sub.loc[~good, ["x", "y"]] = np.nan

    records.append(sub)

df_clean = pd.concat(records, ignore_index=True)
df_clean = df_clean[["frame", "id", "x", "y", "likelihood"]]

df_clean.head()

In [ ]:
df_clean = df_clean.sort_values(["id", "frame"]).copy()

# Interpolate missing positions per mouse
df_clean["x"] = df_clean.groupby("id")["x"].transform(
    lambda s: s.interpolate(limit_direction="both")
)
df_clean["y"] = df_clean.groupby("id")["y"].transform(
    lambda s: s.interpolate(limit_direction="both")
)

# Smooth with a rolling mean per mouse
window = 5
df_clean["x_filt"] = df_clean.groupby("id")["x"].transform(
    lambda s: s.rolling(window=window, center=True, min_periods=1).mean()
)
df_clean["y_filt"] = df_clean.groupby("id")["y"].transform(
    lambda s: s.rolling(window=window, center=True, min_periods=1).mean()
)

df_clean.head()

In [ ]:
df_clean = df_clean.sort_values(["id", "frame"]).copy()

df_clean["dx_px"] = df_clean.groupby("id")["x_filt"].diff().fillna(0.0)
df_clean["dy_px"] = df_clean.groupby("id")["y_filt"].diff().fillna(0.0)

df_clean["dist_px"] = np.sqrt(df_clean["dx_px"]**2 + df_clean["dy_px"]**2)
df_clean["dist_cm"] = df_clean["dist_px"] * PIXEL_TO_CM

max_frame = df_clean["frame"].max()
recording_duration_sec = max_frame / FRAME_RATE
print("Approx recording duration (s):", recording_duration_sec)

activity_df = (
    df_clean.groupby("id")[["dist_px", "dist_cm"]]
    .agg(total_distance_px=("dist_px", "sum"),
         total_distance_cm=("dist_cm", "sum"))
    .reset_index()
)

activity_df["mean_speed_cm_s"] = activity_df["total_distance_cm"] / recording_duration_sec

activity_df

In [ ]:
radius_cm = 6.0
radius_px = radius_cm / PIXEL_TO_CM
print("Social radius (px):", radius_px)

pivot_x = df_clean.pivot(index="frame", columns="id", values="x_filt")
pivot_y = df_clean.pivot(index="frame", columns="id", values="y_filt")

social_counts = {animal: 0 for animal in animals}
total_frames = pivot_x.shape[0]

for frame in pivot_x.index:
    xs = pivot_x.loc[frame].values
    ys = pivot_y.loc[frame].values
    valid = ~(np.isnan(xs) | np.isnan(ys))
    if valid.sum() < 2:
        continue
    xs_v = xs[valid]
    ys_v = ys[valid]
    ids_v = np.array(animals)[valid]

    coords = np.stack([xs_v, ys_v], axis=1)
    diff = coords[:, None, :] - coords[None, :, :]
    dists = np.sqrt(np.sum(diff**2, axis=2))
    np.fill_diagonal(dists, np.inf)

    social_here = (dists < radius_px).any(axis=1)

    for mid, flag in zip(ids_v, social_here):
        if flag:
            social_counts[mid] += 1

social_df = pd.DataFrame({
    "id": list(social_counts.keys()),
    "social_frames": list(social_counts.values())
})
social_df["social_fraction"] = social_df["social_frames"] / total_frames

social_df

In [ ]:
df_clean["speed_cm_frame"] = df_clean["dist_cm"]
df_clean["speed_cm_s"] = df_clean["speed_cm_frame"] * FRAME_RATE

rest_threshold_cm_s = 1.0
df_clean["is_rest"] = df_clean["speed_cm_s"] < rest_threshold_cm_s

rest_counts = df_clean.groupby("id")["is_rest"].sum()
total_counts = df_clean.groupby("id")["is_rest"].count()

previous_df = (rest_counts / total_counts).reset_index(name="previous_metric")

previous_df


In [ ]:
df_clean["r_px"] = np.sqrt(
    (df_clean["x_filt"] - ARENA_CENTER_X) ** 2 +
    (df_clean["y_filt"] - ARENA_CENTER_Y) ** 2
)

wall_threshold = 0.8 * ARENA_RADIUS_PX
df_clean["is_wall"] = df_clean["r_px"] >= wall_threshold

wall_counts = df_clean.groupby("id")["is_wall"].sum()
total_counts2 = df_clean.groupby("id")["is_wall"].count()

custom_df = (wall_counts / total_counts2).reset_index(name="custom_metric")

custom_df

In [ ]:
summary_df = (
    activity_df[["id", "total_distance_px", "total_distance_cm", "mean_speed_cm_s"]]
    .merge(social_df[["id", "social_frames", "social_fraction"]], on="id")
    .merge(previous_df[["id", "previous_metric"]], on="id")
    .merge(custom_df[["id", "custom_metric"]], on="id")
)

summary_df


In [ ]:
scores_df = summary_df.set_index("id").copy()

metric_cols = ["total_distance_cm", "social_fraction", "previous_metric", "custom_metric"]

scores_df_norm = (scores_df[metric_cols] - scores_df[metric_cols].min()) / (
    scores_df[metric_cols].max() - scores_df[metric_cols].min()
)

scores_df_norm = scores_df_norm.rename(columns={
    "total_distance_cm": "activity_score",
    "social_fraction": "social_score",
    "previous_metric": "rest_score",
    "custom_metric": "wall_score"
})

scores_df_norm

scores_df_norm.plot(kind="bar")
plt.ylabel("Score (0 to 1)")
plt.title("Mouse House profiles: activity, sociality, rest, wall occupancy")
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

for col in scores_df_norm.columns:
    print(f"\nMedals for {col}:")
    ranking = scores_df_norm[col].sort_values(ascending=False)
    if len(ranking) == 0:
        continue
    print("Gold:  ", ranking.index[0])
    if len(ranking) > 1:
        print("Silver:", ranking.index[1])
    if len(ranking) > 2:
        print("Bronze:", ranking.index[2])